In [ ]:
# ============================================================
# PROJECT - VaR & ES

# Notes:
# - Historical prices are downloaded with yfinance only.
# ============================================================

# =========================
# IMPORTS
# =========================
import math
import random
import yfinance as yf
import matplotlib.pyplot as plt


# =========================
# SER INPUTS
# =========================
# Portfolio of 2 different assets
TICKER_1 = "XLE"   # Energy Select Sector SPDR Fund
TICKER_2 = "XLK"   # Technology Select Sector SPDR Fund

PORTFOLIO_VALUE_EUR = 18000.0

# Portfolio weights
W1 = 0.35
W2 = 0.65

# One-year period under normal economic conditions
NORMAL_START = "2021-01-01"
NORMAL_END = "2021-12-31"

# One-year stressed period
STRESS_START = "2020-02-01"
STRESS_END = "2020-12-31"

# Risk-free rate used for Black-Scholes
RISK_FREE_RATE = 0.04

# Option parameters for Questions 5 and 7
OPTION_UNDERLYING = TICKER_2      # Call option written on XLK
OPTION_MATURITY_YEARS = 1.0 / 12.0  # 1 month
OPTION_STRIKE_STYLE = "ATM"       

# EWMA / Hybrid lambda
LAMBDA_EWMA = 0.94
LAMBDA_HYBRID = 0.94

# Monte Carlo settings
N_MONTE_CARLO = 20000
RANDOM_SEED = 42

# Confidence levels
ALPHA_VAR_1 = 0.01
ALPHA_VAR_5 = 0.05
ALPHA_VAR_10 = 0.10
ALPHA_ES_5 = 0.05

# One-tailed z-values from the usual normal table
Z_VALUES = {
    0.01: -2.33,
    0.05: -1.65,
    0.10: -1.28
}

# Export settings
EXPORT_DATASET_CSV = True
DATASET_FILENAME = "xle_xlk_portfolio_dataset.csv"


# =========================
# BASIC UTILITY FUNCTIONS
# =========================
def check_weights(w1, w2):
    total = w1 + w2
    if abs(total - 1.0) > 1e-10:
        raise ValueError(f"Weights must sum to 1. Current sum = {total}")


def mean_manual(values):
    if len(values) == 0:
        raise ValueError("Cannot compute the mean of an empty list.")
    total = 0.0
    for x in values:
        total += x
    return total / len(values)


def variance_manual_sample(values):
    """
    Sample variance:
    s^2 = (1 / (N - 1)) * sum((x_i - mean)^2)
    """
    n = len(values)
    if n < 2:
        raise ValueError("At least 2 observations are required for sample variance.")

    mu = mean_manual(values)
    total = 0.0
    for x in values:
        total += (x - mu) ** 2

    return total / (n - 1)


def std_manual_sample(values):
    return math.sqrt(variance_manual_sample(values))


def covariance_manual_sample(x, y):
    n = len(x)

    if n != len(y):
        raise ValueError("The two series must have the same length.")
    if n < 2:
        raise ValueError("At least 2 observations are required for covariance.")

    mx = mean_manual(x)
    my = mean_manual(y)

    total = 0.0
    for i in range(n):
        total += (x[i] - mx) * (y[i] - my)

    return total / (n - 1)


def correlation_manual(x, y):
    sx = std_manual_sample(x)
    sy = std_manual_sample(y)

    if sx == 0 or sy == 0:
        return 0.0

    cov_xy = covariance_manual_sample(x, y)
    return cov_xy / (sx * sy)


def sort_ascending(values):
    copied = values[:]
    copied.sort()
    return copied


def percentile_from_sorted(sorted_vals, alpha):
    if len(sorted_vals) == 0:
        raise ValueError("Cannot compute percentile from an empty list.")

    n = len(sorted_vals)
    k = math.ceil(alpha * n) - 1

    if k < 0:
        k = 0
    if k >= n:
        k = n - 1

    return sorted_vals[k]


def quantile_historical(values, alpha):
    sorted_vals = sort_ascending(values)
    return percentile_from_sorted(sorted_vals, alpha)


def print_line():
    print("=" * 100)


def export_dataset_csv(dates, prices_1, prices_2, returns_1, returns_2, portfolio_returns):
    with open(DATASET_FILENAME, "w", encoding="utf-8") as f:
        f.write(
            f"Date,{TICKER_1}_Price,{TICKER_2}_Price,"
            f"{TICKER_1}_Return,{TICKER_2}_Return,Portfolio_Return\n"
        )

        for i in range(len(portfolio_returns)):
            f.write(
                f"{dates[i]},{prices_1[i]},{prices_2[i]},"
                f"{returns_1[i]},{returns_2[i]},{portfolio_returns[i]}\n"
            )


# =========================
# NORMAL DISTRIBUTION FUNCTIONS
# =========================
def normal_cdf(x):
    """
    Standard normal cumulative distribution function:
    N(x) = 0.5 * [1 + erf(x / sqrt(2))]
    """
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def normal_pdf(x):
    """
    Standard normal probability density function.
    """
    return (1.0 / math.sqrt(2.0 * math.pi)) * math.exp(-0.5 * x * x)


# =========================
# BLACK-SCHOLES FUNCTIONS
# =========================
def black_scholes_call_price(S, K, r, sigma, T):
    """
    Black-Scholes price of a European call option:
    C = S*N(d1) - K*exp(-rT)*N(d2)
    """
    if T <= 0:
        return max(S - K, 0.0)

    if sigma <= 0:
        return max(S - K * math.exp(-r * T), 0.0)

    d1 = (math.log(S / K) + (r + 0.5 * sigma * sigma) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)

    return S * normal_cdf(d1) - K * math.exp(-r * T) * normal_cdf(d2)


def black_scholes_delta_call(S, K, r, sigma, T):
    if T <= 0:
        return 1.0 if S > K else 0.0

    if sigma <= 0:
        return 1.0 if S > K else 0.0

    d1 = (math.log(S / K) + (r + 0.5 * sigma * sigma) * T) / (sigma * math.sqrt(T))
    return normal_cdf(d1)


def black_scholes_gamma_call(S, K, r, sigma, T):
    if T <= 0 or sigma <= 0 or S <= 0:
        return 0.0

    d1 = (math.log(S / K) + (r + 0.5 * sigma * sigma) * T) / (sigma * math.sqrt(T))
    return normal_pdf(d1) / (S * sigma * math.sqrt(T))


# =========================
# RANDOM GENERATOR
# =========================
def random_standard_normal_box_muller():
    """
    Generates one N(0,1) random variable with the Box-Muller method.
    """
    u1 = random.random()
    u2 = random.random()

    if u1 == 0:
        u1 = 1e-12

    z = math.sqrt(-2.0 * math.log(u1)) * math.cos(2.0 * math.pi * u2)
    return z


# =========================
# DATA DOWNLOAD AND PREPARATION
# =========================
def download_adjusted_close(ticker, start_date, end_date):
    data = yf.download(ticker, start=start_date, end=end_date, auto_adjust=True, progress=False)

    if data.empty:
        raise ValueError(f"No data downloaded for {ticker}.")

    closes = data["Close"].squeeze().tolist()
    dates = [str(d.date()) for d in data.index]

    return dates, closes


def align_two_series(dates1, prices1, dates2, prices2):
    """
    Align two price series on common trading dates.
    """
    dict1 = {}
    dict2 = {}

    for i in range(len(dates1)):
        dict1[dates1[i]] = prices1[i]

    for i in range(len(dates2)):
        dict2[dates2[i]] = prices2[i]

    common_dates = []
    for d in dict1:
        if d in dict2:
            common_dates.append(d)

    common_dates.sort()

    p1 = []
    p2 = []
    for d in common_dates:
        p1.append(dict1[d])
        p2.append(dict2[d])

    return common_dates, p1, p2


def simple_returns(prices):
    """
    Simple return:
    r_t = (P_t / P_{t-1}) - 1
    """
    if len(prices) < 2:
        raise ValueError("At least 2 prices are required to compute returns.")

    returns = []
    for i in range(1, len(prices)):
        ret = prices[i] / prices[i - 1] - 1.0
        returns.append(ret)

    return returns


def portfolio_returns_two_assets(r1, r2, w1, w2):
    if len(r1) != len(r2):
        raise ValueError("Return series must have the same length.")

    rp = []
    for i in range(len(r1)):
        rp.append(w1 * r1[i] + w2 * r2[i])

    return rp


# =========================
# PORTFOLIO FORMULAS 
# =========================
def portfolio_expected_return_two_assets(mu1, mu2, w1, w2):
    """
    E(Rp) = w1*mu1 + w2*mu2
    """
    return w1 * mu1 + w2 * mu2


def portfolio_volatility_two_assets(sigma1, sigma2, rho12, w1, w2):
    """
    sigma_p = sqrt(w1^2*sigma1^2 + w2^2*sigma2^2 + 2*w1*w2*rho12*sigma1*sigma2)
    """
    variance_p = (
        (w1 ** 2) * (sigma1 ** 2)
        + (w2 ** 2) * (sigma2 ** 2)
        + 2.0 * w1 * w2 * rho12 * sigma1 * sigma2
    )
    return math.sqrt(variance_p)


# =========================
# VaR / ES FUNCTIONS
# =========================
def var_parametric(mu, sigma, portfolio_value, alpha):
    """
    Parametric VaR:
    VaR_alpha = - (mu + z_alpha * sigma) * V
    """
    z = Z_VALUES[alpha]
    return -(mu + z * sigma) * portfolio_value


def ewma_variance_series(returns, lambda_):
    """
    EWMA variance:
    sigma_t^2 = lambda*sigma_{t-1}^2 + (1-lambda)*r_{t-1}^2
    """
    if len(returns) < 2:
        raise ValueError("At least 2 returns are required for EWMA.")

    initial_var = variance_manual_sample(returns)
    sigma2 = [initial_var]

    for i in range(1, len(returns)):
        new_var = lambda_ * sigma2[-1] + (1.0 - lambda_) * (returns[i - 1] ** 2)
        sigma2.append(new_var)

    return sigma2


def var_parametric_ewma(returns, portfolio_value, alpha, lambda_):
    mu = mean_manual(returns)
    sigma2_series = ewma_variance_series(returns, lambda_)
    sigma_t = math.sqrt(sigma2_series[-1])

    return -(mu + Z_VALUES[alpha] * sigma_t) * portfolio_value, sigma_t


def var_historical(returns, portfolio_value, alpha):
    q = quantile_historical(returns, alpha)
    return -q * portfolio_value, q


def es_historical(returns, portfolio_value, alpha):
    """
    Historical Expected Shortfall:
    average of returns lower than or equal to the alpha-quantile.
    """
    q = quantile_historical(returns, alpha)

    tail = []
    for r in returns:
        if r <= q:
            tail.append(r)

    if len(tail) == 0:
        return 0.0, q

    avg_tail = mean_manual(tail)
    return -avg_tail * portfolio_value, q


def es_parametric_normal(mu, sigma, portfolio_value, alpha):
    """
    Parametric ES under the normality assumption:
    ES_alpha = (-mu + sigma*phi(z_alpha)/alpha) * V
    """
    z = Z_VALUES[alpha]
    phi = normal_pdf(z)
    es = (-mu + sigma * phi / alpha) * portfolio_value
    return es


# =========================
# HYBRID APPROACH
# =========================
def hybrid_weights(n, lambda_):
    """
    Hybrid weights:
    weight_i = [(1-lambda)/(1-lambda^n)] * lambda^(i-1)
    where i = 1 corresponds to the most recent observation.
    """
    if n <= 0:
        raise ValueError("n must be strictly positive.")

    denom = 1.0 - (lambda_ ** n)
    base = (1.0 - lambda_) / denom

    weights = []
    for i in range(1, n + 1):
        w = base * (lambda_ ** (i - 1))
        weights.append(w)

    return weights


def var_hybrid(returns, portfolio_value, alpha, lambda_):
    """
    Hybrid VaR:
    1) assign exponentially decaying weights by recency,
    2) sort returns from worst to best,
    3) cumulate weights until alpha is reached.
    """
    n = len(returns)
    weights_recent_to_old = hybrid_weights(n, lambda_)

    pairs = []
    for i in range(n):
        weight = weights_recent_to_old[n - 1 - i]
        pairs.append((returns[i], weight))

    pairs.sort(key=lambda x: x[0])

    cumulative_weight = 0.0
    threshold_return = pairs[0][0]

    for ret, wt in pairs:
        cumulative_weight += wt
        threshold_return = ret
        if cumulative_weight >= alpha:
            break

    var_value = -threshold_return * portfolio_value
    return var_value, threshold_return, pairs


# =========================
# MONTE CARLO APPROACH
# =========================
def monte_carlo_var_parametric(mu, sigma, portfolio_value, alpha, n_sim):
    """
    Monte Carlo VaR with simulated normal returns.
    """
    simulated_returns = []

    for _ in range(n_sim):
        z = random_standard_normal_box_muller()
        r = mu + sigma * z
        simulated_returns.append(r)

    var_mc, q = var_historical(simulated_returns, portfolio_value, alpha)
    return var_mc, simulated_returns, q


# =========================
# VaR CONVERSIONS
# =========================
def scale_var_sqrt_time(var_daily, horizon_days):
    return var_daily * math.sqrt(horizon_days)


# =========================
# GBM AND TAYLOR SERIES
# =========================
def simulate_gbm_terminal_prices(S0, mu, sigma, T_years, n_paths):
    """
    GBM terminal price:
    S_T = S_0 * exp((mu - 0.5*sigma^2)*T + sigma*sqrt(T)*Z)
    """
    prices = []

    for _ in range(n_paths):
        z = random_standard_normal_box_muller()
        ST = S0 * math.exp((mu - 0.5 * sigma * sigma) * T_years + sigma * math.sqrt(T_years) * z)
        prices.append(ST)

    return prices


def taylor_delta_gamma_pnl(delta, gamma, dS):
    """
    Delta-gamma approximation:
    dC ≈ Delta*dS + 0.5*Gamma*(dS)^2
    """
    return delta * dS + 0.5 * gamma * (dS ** 2)


def var_taylor_delta_gamma(S0, K, r, sigma, T, mu_underlying, n_paths, alpha):
    delta = black_scholes_delta_call(S0, K, r, sigma, T)
    gamma = black_scholes_gamma_call(S0, K, r, sigma, T)

    simulated_prices = simulate_gbm_terminal_prices(S0, mu_underlying, sigma, T, n_paths)
    pnl = []

    for ST in simulated_prices:
        dS = ST - S0
        dC = taylor_delta_gamma_pnl(delta, gamma, dS)
        pnl.append(dC)

    pnl_sorted = sort_ascending(pnl)
    q = percentile_from_sorted(pnl_sorted, alpha)
    var_value = -q

    return var_value, pnl, delta, gamma


# =========================
# FULL REPRICING VaR
# =========================
def full_repricing_var_historical(price_series_underlying, sigma, r, T, K, alpha):
    """
    Historical Full Repricing VaR:
    S^n = S0 * (S_i / S_{i-1})
    C^n = BS(S^n, K, sigma, r, T)
    dC^n = C^n - C0
    """
    if len(price_series_underlying) < 2:
        raise ValueError("At least 2 underlying prices are required.")

    S0 = price_series_underlying[-1]
    C0 = black_scholes_call_price(S0, K, r, sigma, T)

    pnl = []
    shocked_prices = []
    option_prices = []

    for i in range(1, len(price_series_underlying)):
        shock_factor = price_series_underlying[i] / price_series_underlying[i - 1]
        Sn = S0 * shock_factor
        Cn = black_scholes_call_price(Sn, K, r, sigma, T)
        dC = Cn - C0

        shocked_prices.append(Sn)
        option_prices.append(Cn)
        pnl.append(dC)

    pnl_sorted = sort_ascending(pnl)
    q = percentile_from_sorted(pnl_sorted, alpha)
    var_value = -q

    return var_value, pnl, shocked_prices, option_prices, C0

In [ ]:

# =========================
# PLOTTING FUNCTIONS
# =========================
def plot_histogram(data, bins, title, xlabel, threshold=None, filename=None):
    plt.figure(figsize=(10, 6))
    plt.hist(data, bins=bins, edgecolor="black")

    if threshold is not None:
        plt.axvline(threshold, linestyle="--", linewidth=2, label=f"Threshold = {threshold:.6f}")
        plt.legend()

    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Frequency")
    plt.tight_layout()

    if filename is not None:
        plt.savefig(filename, dpi=200)

    plt.show()


def plot_ewma_volatility(sigma2_series, filename=None):
    sigma_series = []
    for x in sigma2_series:
        sigma_series.append(math.sqrt(x))

    plt.figure(figsize=(10, 6))
    plt.plot(sigma_series)
    plt.title(f"EWMA Volatility Series - {TICKER_1}/{TICKER_2} Portfolio")
    plt.xlabel("Time")
    plt.ylabel("Volatility")
    plt.tight_layout()

    if filename is not None:
        plt.savefig(filename, dpi=200)

    plt.show()


In [ ]:
# =========================
# MAIN PROGRAM
# =========================
def main():
    random.seed(RANDOM_SEED)
    check_weights(W1, W2)

    print_line()
    print("PROJECT - VaR & ES")
    print("2-ASSET PORTFOLIO:", TICKER_1, "AND", TICKER_2)
    print_line()

    # ------------------------------------------------------------
    # DOWNLOAD NORMAL PERIOD DATA
    # ------------------------------------------------------------
    print("\nSECTION A - DOWNLOAD OF HISTORICAL DATA (NORMAL PERIOD)")

    dates1, prices1 = download_adjusted_close(TICKER_1, NORMAL_START, NORMAL_END)
    dates2, prices2 = download_adjusted_close(TICKER_2, NORMAL_START, NORMAL_END)

    dates_common, p1, p2 = align_two_series(dates1, prices1, dates2, prices2)

    r1 = simple_returns(p1)
    r2 = simple_returns(p2)
    return_dates = dates_common[1:]
    rp = portfolio_returns_two_assets(r1, r2, W1, W2)

    if EXPORT_DATASET_CSV:
        export_dataset_csv(return_dates, p1[1:], p2[1:], r1, r2, rp)
        print(f"Dataset exported to: {DATASET_FILENAME}")

    print(f"Asset 1 = {TICKER_1}")
    print(f"Asset 2 = {TICKER_2}")
    print(f"Normal period = {NORMAL_START} to {NORMAL_END}")
    print(f"Number of return observations = {len(rp)}")


    # ------------------------------------------------------------
    # SECTION B - QUESTION 1
    # ------------------------------------------------------------
    print("\nSECTION B - QUESTION 1: CONSTRUCTION OF THE 2-ASSET PORTFOLIO")

    mu1 = mean_manual(r1)
    mu2 = mean_manual(r2)

    sigma1 = std_manual_sample(r1)
    sigma2 = std_manual_sample(r2)
    rho12 = correlation_manual(r1, r2)

    mu_portfolio_annex2 = portfolio_expected_return_two_assets(mu1, mu2, W1, W2)
    sigma_portfolio_annex2 = portfolio_volatility_two_assets(sigma1, sigma2, rho12, W1, W2)

    mu_portfolio_direct = mean_manual(rp)
    sigma_portfolio_direct = std_manual_sample(rp)

    print(f"Portfolio value = {PORTFOLIO_VALUE_EUR:.2f} EUR")
    print(f"Weight of {TICKER_1} = {W1:.4f}")
    print(f"Weight of {TICKER_2} = {W2:.4f}")
    print(f"Mean daily return of {TICKER_1} = {mu1:.8f}")
    print(f"Mean daily return of {TICKER_2} = {mu2:.8f}")
    print(f"Daily volatility of {TICKER_1} = {sigma1:.8f}")
    print(f"Daily volatility of {TICKER_2} = {sigma2:.8f}")
    print(f"Correlation between assets = {rho12:.8f}")

    print("\nPortfolio statistics based on the 2-asset formulas:")
    print(f"Expected daily return = {mu_portfolio_annex2:.8f}")
    print(f"Daily volatility = {sigma_portfolio_annex2:.8f}")

    print("\nPortfolio statistics computed from the portfolio return series:")
    print(f"Expected daily return = {mu_portfolio_direct:.8f}")
    print(f"Daily volatility = {sigma_portfolio_direct:.8f}")


    # ------------------------------------------------------------
    # SECTION C - QUESTION 2a
    # ------------------------------------------------------------
    print("\nSECTION C - QUESTION 2a: PARAMETRIC VaR USING SMA")

    var_sma_1 = var_parametric(
        mu_portfolio_direct,
        sigma_portfolio_direct,
        PORTFOLIO_VALUE_EUR,
        ALPHA_VAR_1
    )

    print(f"Daily parametric VaR (1%) using SMA = {var_sma_1:.2f} EUR")

    # ------------------------------------------------------------
    # SECTION D - QUESTION 2b
    # ------------------------------------------------------------
    print("\nSECTION D - QUESTION 2b: PARAMETRIC VaR USING EWMA")

    var_ewma_1, sigma_ewma = var_parametric_ewma(
        rp,
        PORTFOLIO_VALUE_EUR,
        ALPHA_VAR_1,
        LAMBDA_EWMA
    )

    print(f"Chosen lambda = {LAMBDA_EWMA}")
    print(f"EWMA volatility = {sigma_ewma:.8f}")
    print(f"Daily parametric VaR (1%) using EWMA = {var_ewma_1:.2f} EUR")

    ewma_sigma2_series = ewma_variance_series(rp, LAMBDA_EWMA)
    plot_ewma_volatility(ewma_sigma2_series, filename="xle_xlk_ewma_volatility.png")

    # ------------------------------------------------------------
    # SECTION E - QUESTION 2c
    # ------------------------------------------------------------
    print("\nSECTION E - QUESTION 2c: NON-PARAMETRIC HISTORICAL VaR")

    var_hist_1, q_hist_1 = var_historical(rp, PORTFOLIO_VALUE_EUR, ALPHA_VAR_1)

    print(f"Historical daily VaR (1%) = {var_hist_1:.2f} EUR")
    print(f"Historical return threshold at 1% = {q_hist_1:.8f}")

    plot_histogram(
        rp,
        bins=40,
        title=f"Historical Return Distribution - {TICKER_1}/{TICKER_2} Portfolio",
        xlabel="Daily portfolio return",
        threshold=q_hist_1,
        filename="xle_xlk_historical_var_histogram.png"
    )

    # ------------------------------------------------------------
    # SECTION F - QUESTION 2d
    # ------------------------------------------------------------
    print("\nSECTION F - QUESTION 2d: HYBRID VaR")

    var_hyb_1, q_hyb_1, _ = var_hybrid(rp, PORTFOLIO_VALUE_EUR, ALPHA_VAR_1, LAMBDA_HYBRID)

    print(f"Chosen lambda = {LAMBDA_HYBRID}")
    print(f"Hybrid daily VaR (1%) = {var_hyb_1:.2f} EUR")
    print(f"Hybrid return threshold at 1% = {q_hyb_1:.8f}")


    # ------------------------------------------------------------
    # SECTION G - QUESTION 2e
    # ------------------------------------------------------------
    print("\nSECTION G - QUESTION 2e: MONTE CARLO VaR")

    var_mc_1, sim_returns_mc, q_mc_1 = monte_carlo_var_parametric(
        mu_portfolio_direct,
        sigma_portfolio_direct,
        PORTFOLIO_VALUE_EUR,
        ALPHA_VAR_1,
        N_MONTE_CARLO
    )

    print(f"Number of simulations = {N_MONTE_CARLO}")
    print(f"Monte Carlo daily VaR (1%) = {var_mc_1:.2f} EUR")

    plot_histogram(
        sim_returns_mc,
        bins=50,
        title=f"Monte Carlo Simulated Returns - {TICKER_1}/{TICKER_2} Portfolio",
        xlabel="Simulated daily portfolio return",
        threshold=q_mc_1,
        filename="xle_xlk_monte_carlo_histogram.png"
    )

    # ------------------------------------------------------------
    # SECTION H - COMPARISON OF METHODS
    # ------------------------------------------------------------
    print("\nSECTION H - COMPARISON OF THE DIFFERENT VaR APPROACHES")

    print(f"SMA Parametric VaR (1%)  = {var_sma_1:.2f} EUR")
    print(f"EWMA Parametric VaR (1%) = {var_ewma_1:.2f} EUR")
    print(f"Historical VaR (1%)      = {var_hist_1:.2f} EUR")
    print(f"Hybrid VaR (1%)          = {var_hyb_1:.2f} EUR")
    print(f"Monte Carlo VaR (1%)     = {var_mc_1:.2f} EUR")

    # ------------------------------------------------------------
    # SECTION I - QUESTION 3
    # ------------------------------------------------------------
    print("\nSECTION I - QUESTION 3: VaR CONVERSIONS BASED ON PARAMETRIC SMA")

    var_daily_1 = var_sma_1
    var_weekly_1 = scale_var_sqrt_time(var_daily_1, 5)
    var_monthly_1 = scale_var_sqrt_time(var_daily_1, 21)
    var_annual_1 = scale_var_sqrt_time(var_daily_1, 252)

    var_daily_10 = var_parametric(
        mu_portfolio_direct,
        sigma_portfolio_direct,
        PORTFOLIO_VALUE_EUR,
        ALPHA_VAR_10
    )
    var_weekly_10 = scale_var_sqrt_time(var_daily_10, 5)
    var_monthly_10 = scale_var_sqrt_time(var_daily_10, 21)
    var_annual_10 = scale_var_sqrt_time(var_daily_10, 252)

    print(f"Daily VaR (1%)   = {var_daily_1:.2f} EUR")
    print(f"Weekly VaR (1%)  = {var_weekly_1:.2f} EUR")
    print(f"Monthly VaR (1%) = {var_monthly_1:.2f} EUR")
    print(f"Annual VaR (1%)  = {var_annual_1:.2f} EUR")

    print(f"\nDaily VaR (10%)   = {var_daily_10:.2f} EUR")
    print(f"Weekly VaR (10%)  = {var_weekly_10:.2f} EUR")
    print(f"Monthly VaR (10%) = {var_monthly_10:.2f} EUR")
    print(f"Annual VaR (10%)  = {var_annual_10:.2f} EUR")

    # ------------------------------------------------------------
    # SECTION J - QUESTION 4
    # ------------------------------------------------------------
    print("\nSECTION J - QUESTION 4: STRESSED HISTORICAL VaR")

    stress_dates1, stress_prices1 = download_adjusted_close(TICKER_1, STRESS_START, STRESS_END)
    stress_dates2, stress_prices2 = download_adjusted_close(TICKER_2, STRESS_START, STRESS_END)

    stress_dates_common, stress_p1, stress_p2 = align_two_series(
        stress_dates1, stress_prices1, stress_dates2, stress_prices2
    )

    stress_r1 = simple_returns(stress_p1)
    stress_r2 = simple_returns(stress_p2)
    stress_rp = portfolio_returns_two_assets(stress_r1, stress_r2, W1, W2)

    stressed_var_hist_1, _ = var_historical(stress_rp, PORTFOLIO_VALUE_EUR, ALPHA_VAR_1)

    print(f"Stressed period = {STRESS_START} to {STRESS_END}")
    print(f"Historical VaR (1%) in normal period   = {var_hist_1:.2f} EUR")
    print(f"Historical Stressed VaR (1%)           = {stressed_var_hist_1:.2f} EUR")
    print(f"Difference                             = {stressed_var_hist_1 - var_hist_1:.2f} EUR")

    # ------------------------------------------------------------
    # SECTION K - QUESTION 6
    # ------------------------------------------------------------
    print("\nSECTION K - QUESTION 6: EXPECTED SHORTFALL (5%)")

    es_param_5 = es_parametric_normal(
        mu_portfolio_direct,
        sigma_portfolio_direct,
        PORTFOLIO_VALUE_EUR,
        ALPHA_ES_5
    )

    es_hist_5, _ = es_historical(
        rp,
        PORTFOLIO_VALUE_EUR,
        ALPHA_ES_5
    )

    var_param_5 = var_parametric(
        mu_portfolio_direct,
        sigma_portfolio_direct,
        PORTFOLIO_VALUE_EUR,
        ALPHA_VAR_5
    )

    var_hist_5, _ = var_historical(
        rp,
        PORTFOLIO_VALUE_EUR,
        ALPHA_VAR_5
    )

    print(f"Parametric VaR (5%) = {var_param_5:.2f} EUR")
    print(f"Parametric ES  (5%) = {es_param_5:.2f} EUR")
    print(f"Historical VaR (5%) = {var_hist_5:.2f} EUR")
    print(f"Historical ES  (5%) = {es_hist_5:.2f} EUR")

    # ------------------------------------------------------------
    # SECTION L - QUESTION 5
    # ------------------------------------------------------------
    print("\nSECTION L - QUESTION 5: TAYLOR SERIES VaR FOR THE CALL OPTION")

    if OPTION_UNDERLYING == TICKER_1:
        underlying_prices = p1
        underlying_returns = r1
    else:
        underlying_prices = p2
        underlying_returns = r2

    S0_option = underlying_prices[-1]
    sigma_underlying = std_manual_sample(underlying_returns) * math.sqrt(252.0)
    mu_underlying = mean_manual(underlying_returns) * 252.0

    if OPTION_STRIKE_STYLE == "ATM":
        K_option = S0_option
    else:
        K_option = OPTION_STRIKE_CUSTOM

    taylor_var_1, taylor_pnl, delta_option, gamma_option = var_taylor_delta_gamma(
        S0=S0_option,
        K=K_option,
        r=RISK_FREE_RATE,
        sigma=sigma_underlying,
        T=OPTION_MATURITY_YEARS,
        mu_underlying=mu_underlying,
        n_paths=N_MONTE_CARLO,
        alpha=ALPHA_VAR_1
    )

    print(f"Underlying asset for the option = {OPTION_UNDERLYING}")
    print(f"Current underlying price S0 = {S0_option:.4f}")
    print(f"Strike price K = {K_option:.4f}")
    print(f"Annualized volatility = {sigma_underlying:.6f}")
    print(f"Delta = {delta_option:.6f}")
    print(f"Gamma = {gamma_option:.6f}")
    print(f"Taylor delta-gamma VaR (1%) = {taylor_var_1:.6f} EUR per option")

    plot_histogram(
        taylor_pnl,
        bins=50,
        title=f"Option P&L - Taylor Delta-Gamma Approximation ({OPTION_UNDERLYING})",
        xlabel="Option P&L",
        threshold=-taylor_var_1,
        filename="xlk_taylor_option_histogram.png"
    )

    # ------------------------------------------------------------
    # SECTION M - QUESTION 7
    # ------------------------------------------------------------
    print("\nSECTION M - QUESTION 7: FULL REPRICING VaR FOR THE CALL OPTION")

    full_repricing_var_1, full_pnl, shocked_prices, shocked_option_prices, C0 = full_repricing_var_historical(
        price_series_underlying=underlying_prices,
        sigma=sigma_underlying,
        r=RISK_FREE_RATE,
        T=OPTION_MATURITY_YEARS,
        K=K_option,
        alpha=ALPHA_VAR_1
    )

    print(f"Current option price C0 = {C0:.6f}")
    print(f"Full Repricing VaR (1%) = {full_repricing_var_1:.6f} EUR per option")

    plot_histogram(
        full_pnl,
        bins=40,
        title=f"Option P&L - Full Repricing Historical VaR ({OPTION_UNDERLYING})",
        xlabel="Option P&L",
        threshold=-full_repricing_var_1,
        filename="xlk_full_repricing_histogram.png"
    )

    print("\nComparison of option VaR measures:")
    print(f"Taylor Delta-Gamma VaR (1%) = {taylor_var_1:.6f} EUR")
    print(f"Full Repricing VaR (1%)     = {full_repricing_var_1:.6f} EUR")


    # ------------------------------------------------------------
    # SECTION N - FINAL SUMMARY
    # ------------------------------------------------------------
    print("\nSECTION N - FINAL SUMMARY")
    print_line()
    print("PORTFOLIO RESULTS")
    print_line()
    print(f"Portfolio value                         = {PORTFOLIO_VALUE_EUR:.2f} EUR")
    print(f"Daily expected return                   = {mu_portfolio_direct:.8f}")
    print(f"Daily volatility                        = {sigma_portfolio_direct:.8f}")
    print(f"Parametric VaR SMA (1%)                 = {var_sma_1:.2f} EUR")
    print(f"Parametric VaR EWMA (1%)                = {var_ewma_1:.2f} EUR")
    print(f"Historical VaR (1%)                     = {var_hist_1:.2f} EUR")
    print(f"Hybrid VaR (1%)                         = {var_hyb_1:.2f} EUR")
    print(f"Monte Carlo VaR (1%)                    = {var_mc_1:.2f} EUR")
    print(f"Historical Stressed VaR (1%)            = {stressed_var_hist_1:.2f} EUR")
    print(f"Parametric ES (5%)                      = {es_param_5:.2f} EUR")
    print(f"Historical ES (5%)                      = {es_hist_5:.2f} EUR")
    print_line()
    print("OPTION RESULTS")
    print_line()
    print(f"Taylor Delta-Gamma VaR (1%)             = {taylor_var_1:.6f} EUR")
    print(f"Full Repricing VaR (1%)                 = {full_repricing_var_1:.6f} EUR")
    print_line()


